# Can we predict late deliveries?
Goal: predict `late_delivery` at **dispatch time** from consignment attributes.

Key decision up front: `claim_paid_eur` is compensation paid on a claim *settled for the consignment*, i.e. it is only known after delivery. It is a consequence of lateness, not a predictor (checked below), so it is excluded from the model.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, RocCurveDisplay
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, 'unique ids:', df.consignment.is_unique, '| missing cells:', df.isna().sum().sum())
print(df.dispatched_on.min(), '->', df.dispatched_on.max())
df.describe().T

## Target and leakage check

In [ ]:
print(df.late_delivery.value_counts(normalize=True).rename('share'))
print(df.groupby('late_delivery').claim_paid_eur.agg(mean='mean', share_with_claim=lambda s: (s>0).mean()))
print('AUC of claim_paid_eur alone (illustration only):', round(roc_auc_score(df.late_delivery, df.claim_paid_eur),3))

Every late consignment has a claim and almost no on-time one does. Using it would give a near-perfect but useless model, since the claim does not exist when the prediction is needed. Dropped.

## Exploration

In [ ]:
df['month'] = df.dispatched_on.dt.to_period('M')
fig, ax = plt.subplots(1, 3, figsize=(15,4))
df.groupby('carrier').late_delivery.mean().plot.bar(ax=ax[0], title='Late rate by carrier')
df.groupby(pd.qcut(df.distance_km, 10, duplicates='drop'), observed=True).late_delivery.mean().plot(ax=ax[1], marker='o', title='Late rate by distance decile'); ax[1].tick_params(axis='x', rotation=60)
df.groupby('month').late_delivery.mean().plot(ax=ax[2], marker='o', title='Late rate by dispatch month'); ax[2].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 3, figsize=(15,4))
df.groupby(df.dispatched_on.dt.hour).late_delivery.mean().plot(ax=ax[0], marker='o', title='by hour')
df.groupby(df.dispatched_on.dt.dayofweek).late_delivery.mean().plot(ax=ax[1], marker='o', title='by weekday (0=Mon)')
df.groupby('parcels').late_delivery.mean().plot(ax=ax[2], marker='o', title='by parcels')
plt.tight_layout(); plt.show()

## Label drift: the late rate collapses after September 2025
The late rate swings between ~10% and ~50% through Sep 2025, then drops to 1-5% for every month from Oct 2025. Either delivery performance genuinely changed, or the labels for recent consignments are incomplete (e.g. late deliveries not yet recorded). The data cannot tell us which.

Consequence: a naive "last 20%" test set lands entirely in the collapsed regime (2.2% late), so it would measure drift rather than skill. The model is therefore built and tested on consignments dispatched **before 2025-10-01**, and the later period is scored separately as a stress test.

In [ ]:
m = df.groupby('month').agg(n=('late_delivery','size'), late_rate=('late_delivery','mean'), share_with_claim=('claim_paid_eur', lambda s: (s>0).mean()))
m.round(3)

## Features and time-ordered evaluation
Only dispatch-time information is used. Within the stable period (before 2025-10-01), the test set is the **last 20% by dispatch date**; model selection uses forward-chaining CV on the earlier 80%.

In [ ]:
df['hour'] = df.dispatched_on.dt.hour
df['dow'] = df.dispatched_on.dt.dayofweek
num = ['distance_km','weight_kg','parcels','hour','dow']
cat = ['carrier']
stable = df[df.dispatched_on < '2025-10-01']
recent = df[df.dispatched_on >= '2025-10-01']
X, y = stable[num+cat], stable.late_delivery
cut = int(len(stable)*0.8)
Xtr, Xte, ytr, yte = X[:cut], X[cut:], y[:cut], y[cut:]
print('train', Xtr.shape, ytr.mean().round(3), '| test', Xte.shape, yte.mean().round(3))

pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(), cat)])
models = {
 'baseline (prior)': DummyClassifier(strategy='prior'),
 'logistic': Pipeline([('p', pre), ('m', LogisticRegression(max_iter=1000))]),
 'grad. boosting': Pipeline([('p', pre), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
for n, m in models.items():
    s = cross_val_score(m, Xtr, ytr, cv=TimeSeriesSplit(5), scoring='roc_auc')
    print(f'{n:18s} CV AUC {s.mean():.3f} +/- {s.std():.3f}')

## Held-out (future) test set

In [ ]:
rows, fitted = [], {}
for n, m in models.items():
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:,1]; fitted[n] = (m, p)
    rows.append(dict(model=n, AUC=roc_auc_score(yte,p), PR_AUC=average_precision_score(yte,p), brier=brier_score_loss(yte,p)))
print('test positive rate (= PR-AUC of a no-skill model):', yte.mean().round(3))
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
for n,(m,p) in fitted.items(): RocCurveDisplay.from_predictions(yte, p, name=n, ax=ax)
plt.show()
p = fitted['grad. boosting'][1]
q = pd.qcut(p, 5, duplicates='drop')
pd.DataFrame({'late':yte.values,'bin':q}).groupby('bin', observed=True).late.agg(['mean','size']).rename(columns={'mean':'actual late rate'})

### Stress test on the post-Sep-2025 period
Models trained on the stable period, scored on later consignments. Ranking (AUC) may hold up even if the base rate does not.

In [ ]:
rows = []
for n, mdl in fitted.items():
    pr = mdl[0].predict_proba(recent[num+cat])[:,1]
    rows.append(dict(model=n, AUC=roc_auc_score(recent.late_delivery, pr), mean_predicted=pr.mean(), actual_late_rate=recent.late_delivery.mean()))
pd.DataFrame(rows).set_index('model').round(3)

## What drives it?

In [ ]:
m = fitted['grad. boosting'][0]
pi = permutation_importance(m, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, index=Xte.columns); err = pd.Series(pi.importances_std, index=Xte.columns)
order = imp.sort_values().index
imp[order].plot.barh(xerr=err[order], title='Permutation importance (drop in test AUC)'); plt.show()
lr = fitted['logistic'][0]
pd.Series(lr.named_steps['m'].coef_[0], index=lr.named_steps['p'].get_feature_names_out()).round(3).sort_values()

## Conclusions
- **Yes, partly.** On a time-ordered hold-out inside the stable period (dispatched before 2025-10-01), gradient boosting reaches AUC ~0.73 and logistic regression ~0.71 (no-skill 0.50). Precision-recall AUC is ~0.53 against a 0.32 base rate, and the top-fifth risk bin is late ~56% of the time versus ~9% in the bottom fifth. That is useful for ranking or flagging risky consignments, but far from reliable per-consignment prediction.
- **Drivers:** longer distance and heavier weight raise risk; `coastal` is the riskiest carrier, `express` the safest. Hour, weekday and parcel count add little.
- **Leakage avoided:** `claim_paid_eur` predicts lateness almost perfectly (AUC ~0.99) only because claims follow late deliveries; it is unavailable at dispatch and was excluded.
- **Caveat, important:** the late rate falls from ~20-50% to 1-5% from Oct 2025 onward. We could not tell whether this is a real improvement or incomplete labels. The models still rank consignments reasonably there (AUC ~0.69-0.72) but their predicted rate (~23%) is nowhere near the observed ~2.5%, so probabilities are not calibrated for that period. Confirm with the data owner before relying on the recent labels or deploying.
- **Next steps:** resolve the label question; obtain richer features (destination region, promised lead time, weather/holiday, carrier load) since the current ones cap performance; recalibrate periodically.